# SDXL Convert Phase 2-4a (TPU v5e-8, MNN CLIP + VAE full + UNet converter only)

Stops after UNet qairt-converter → saves intermediate artifacts for B2.
Rationale: empirically B1 finishes in ~1h48m which fits under Kaggle 2h TPU-idle window even
if heartbeat fails. B2 (UNet quantizer + context-bin, ~5h) needs heartbeat protection.


In [ ]:
CIVITAI_VERSION_ID = '__CIVITAI_VERSION_ID__'
MODEL_NAME         = '__MODEL_NAME__'
REALISTIC          = '__REALISTIC__' == 'true'
MIN_SOC            = '__MIN_SOC__'
BURN_TPU           = '__BURN_TPU__' == 'true'
print(f'civitai={CIVITAI_VERSION_ID} name={MODEL_NAME} realistic={REALISTIC} min_soc={MIN_SOC} burn_tpu={BURN_TPU}')

In [ ]:
!free -h
!nproc
!df -h /kaggle/working /tmp
!cat /etc/os-release | head -5
!ldd --version | head -1
!grep -m1 'model name' /proc/cpuinfo
!grep -m1 'cpu MHz' /proc/cpuinfo
!lscpu | grep -E 'Model name|Socket|Thread|Core|Vendor|Flags' | head -10

In [ ]:
# Locate Phase 1 output
import glob, os
candidates = glob.glob('/kaggle/input/**/data_sdxl.pkl', recursive=True)
assert candidates, 'Phase 1 data_sdxl.pkl not found in /kaggle/input/ — did Phase 1 kernel complete?'
PHASE1_DIR = os.path.dirname(candidates[0])
print(f'Phase 1 dir: {PHASE1_DIR}')
!ls -lh $PHASE1_DIR

In [ ]:
import os
os.environ['MIN_SOC'] = MIN_SOC
os.environ['MODEL_NAME'] = MODEL_NAME

In [ ]:
# Install tools + convertsdxl + shell helper
import os, subprocess

def _run(cmd, log=None, check=True):
    """bash with errexit + pipefail; pipe to tee if log. Raises on non-zero."""
    if log:
        os.makedirs(os.path.dirname(log), exist_ok=True)
        wrapped = f'set -eo pipefail; {cmd} 2>&1 | tee {log}'
    else:
        wrapped = f'set -eo pipefail; {cmd}'
    rc = subprocess.call(['bash', '-c', wrapped])
    if check and rc != 0:
        raise RuntimeError(f'shell failed (rc={rc}): {cmd[:200]}  log={log}')
    return rc

# Ensure apt cache is fresh before any installs
_run('apt-get update -y > /dev/null')
_run('apt-get install -y unzip zip curl time > /dev/null')
# libc++/libc++abi/libunwind needed by QNN SDK libPyIrGraph.so / libQnnHtp.so — Kaggle Debian 13 trixie doesn't ship them
# Specific LLVM runtime packages — generic libc++-dev/libunwind-dev do NOT provide libunwind.so.1 (GNU libunwind ships .so.8)
_run('apt-get install -y libc++1-19 libc++abi1-19 libunwind-19 > /dev/null')
_run('ldconfig')  # refresh ld cache so newly installed libs are found by ldd / dlopen
_run('curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null 2>&1')
os.environ['PATH'] = f"/root/.local/bin:{os.environ['PATH']}"

_run('rm -rf /tmp/convertsdxl.zip /tmp/convertsdxl_unzip')
_run("curl -sL --fail --retry 5 --retry-delay 5 -o /tmp/convertsdxl.zip 'https://chino.icu/local-dream/convertsdxl.zip'")
_run('mkdir -p /tmp/convertsdxl_unzip')
_run('unzip -q /tmp/convertsdxl.zip -d /tmp/convertsdxl_unzip')

root = subprocess.check_output(
    "find /tmp/convertsdxl_unzip -maxdepth 3 -name 'export_sdxl.sh' -printf '%h\\n' | head -1",
    shell=True, text=True).strip()
assert root, 'export_sdxl.sh not found in unzipped convertsdxl'
NPUCONVERT_DIR = os.path.abspath(root)
os.environ['NPUCONVERT_DIR'] = NPUCONVERT_DIR
print(f'NPUCONVERT_DIR: {NPUCONVERT_DIR}')

In [ ]:
# QNN SDK
import os, subprocess
_run('mkdir -p /tmp/qnn_sdk')
_run(
    "curl -sL --fail --retry 5 --retry-delay 5 -A 'Mozilla/5.0' "
    "-o /tmp/qnn_sdk/qnn.zip "
    "'https://apigwx-aws.qualcomm.com/qsc/public/v1/api/download/software/qualcomm_neural_processing_sdk/v2.28.0.241029.zip'"
)
_run('cd /tmp/qnn_sdk && unzip -q qnn.zip')
bin_file = subprocess.check_output(
    'find /tmp/qnn_sdk -type f -name envsetup.sh -print -quit',
    shell=True, text=True).strip()
assert bin_file, 'envsetup.sh not found in QNN SDK extract'
QNN_SDK_BIN = os.path.dirname(bin_file)
QNN_SDK_ROOT_DIR = os.path.dirname(QNN_SDK_BIN)
os.environ['QNN_SDK_BIN'] = QNN_SDK_BIN
os.environ['QNN_SDK_ROOT_DIR'] = QNN_SDK_ROOT_DIR
print(f'QNN_SDK_ROOT: {QNN_SDK_ROOT_DIR}')

In [ ]:
# Resolve model.safetensors from Phase 1 kernel_sources mount (read-only, no re-download)
import os, glob
candidates = glob.glob('/kaggle/input/**/model.safetensors', recursive=True)
assert candidates, 'model.safetensors not found in /kaggle/input/ — Phase 1 kernel_sources not mounted?'
os.environ['MODEL_PATH'] = candidates[0]
size = os.path.getsize(os.environ['MODEL_PATH'])
assert size > int(1e9), f'model.safetensors truncated: {size} bytes'
print(f'MODEL_PATH -> {os.environ["MODEL_PATH"]} ({size/1e9:.2f} GB)')

In [ ]:
# Python env
import os
os.chdir(os.environ['NPUCONVERT_DIR'])
_run('uv venv -p 3.10.17 --clear')
_run('. .venv/bin/activate && uv sync')

In [ ]:
# Copy Phase 1 output into convert dir
import os, shutil, glob
pkl_list = glob.glob('/kaggle/input/**/data_sdxl.pkl', recursive=True)
assert pkl_list, 'data_sdxl.pkl not found in /kaggle/input/ — Phase 1 mount missing'
pkl = pkl_list[0]
p1_dir = os.path.dirname(pkl)
shutil.copy(pkl, f'{os.environ["NPUCONVERT_DIR"]}/data_sdxl.pkl')
src_img = f'{p1_dir}/images_sdxl'
assert os.path.isdir(src_img), f'{src_img} missing — Phase 1 did not produce calibration images'
dst_img = f'{os.environ["NPUCONVERT_DIR"]}/images_sdxl'
if os.path.exists(dst_img):
    shutil.rmtree(dst_img)
shutil.copytree(src_img, dst_img)
assert len(os.listdir(dst_img)) > 0, f'{dst_img} empty after copy'
_run(f'ls -lh {os.environ["NPUCONVERT_DIR"]}/data_sdxl.pkl')
_run(f'ls {dst_img} | head -5')

In [ ]:
# Mem watcher + TPU heartbeat (gated by BURN_TPU flag set in cell 1).
# Kaggle policy kills TPU VMs idle > 2h. For >2h notebooks (B2 UNet quantizer 4h) this is required.
# For <2h notebooks (B1 ~1h48m) can set burn_tpu=false to skip (saves TF-TPU install + some CPU).
import subprocess, os, time, threading
os.makedirs('/kaggle/working/logs', exist_ok=True)

# --- mem watcher always runs (cheap, writes csv) ---
watcher_sh = '''#!/bin/bash
echo "epoch,datetime,MemAvail_MB,TopRSS_MB,TopProc"
while true; do
  epoch=$(date +%s); dt=$(date -Iseconds)
  mem=$(awk '/MemAvailable:/{print int($2/1024)}' /proc/meminfo)
  line=$(ps -eo rss,comm --sort=-rss --no-headers 2>/dev/null | head -1)
  rss=$(echo "$line" | awk '{print int($1/1024)}')
  proc=$(echo "$line" | awk '{print $2}')
  echo "$epoch,$dt,$mem,$rss,$proc"
  sleep 10
done
'''
open('/tmp/mem_watch.sh','w').write(watcher_sh)
os.chmod('/tmp/mem_watch.sh', 0o755)
p = subprocess.Popen(['/tmp/mem_watch.sh'], stdout=open('/kaggle/working/logs/mem-trace.csv','w'), stderr=subprocess.DEVNULL)
print(f'mem-watcher pid={p.pid}')

if not BURN_TPU:
    print('BURN_TPU=false → skipping TPU heartbeat install+thread. Kaggle may kill this kernel at 2h TPU-idle.')
else:
    # Install TF-TPU (Kaggle official v5e-8 stack) — uninstall default jax first
    _run('export PATH="${HOME}/.local/bin:${PATH}" && uv pip uninstall --system jax 2>&1 | tail -3', check=False)
    _run('export PATH="${HOME}/.local/bin:${PATH}" && uv pip install --system --quiet '
         'tensorflow-tpu==2.18.0 '
         '--find-links https://storage.googleapis.com/libtpu-tf-releases/index.html')

    import tensorflow as tf
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver(tpu="local")
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
    NR = strategy.num_replicas_in_sync
    print(f'Jupyter kernel TPU replicas: {NR}')
    assert NR > 0

    with strategy.scope():
        _model = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(32,32,3)),
            tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
            tf.keras.layers.MaxPool2D(),
            tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
            tf.keras.layers.GlobalAveragePooling2D(),
            tf.keras.layers.Dense(10),
        ])
        _opt = tf.keras.optimizers.Adam(1e-3)

    BATCH_PER_REPLICA = 128
    GLOBAL_BATCH = BATCH_PER_REPLICA * NR

    @tf.function
    def _train_step(x, y):
        with tf.GradientTape() as tape:
            logits = _model(x, training=True)
            loss = tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(y, logits))
        grads = tape.gradient(loss, _model.trainable_variables)
        _opt.apply_gradients(zip(grads, _model.trainable_variables))
        return loss

    def _dataset():
        def _gen():
            while True:
                x = tf.random.normal((GLOBAL_BATCH, 32, 32, 3))
                y = tf.random.uniform((GLOBAL_BATCH,), maxval=10, dtype=tf.int64)
                yield x, y
        ds = tf.data.Dataset.from_generator(
            _gen,
            output_signature=(
                tf.TensorSpec(shape=(GLOBAL_BATCH,32,32,3), dtype=tf.float32),
                tf.TensorSpec(shape=(GLOBAL_BATCH,), dtype=tf.int64),
            ))
        return strategy.experimental_distribute_dataset(ds)

    _stop = threading.Event()
    _step_count = [0]
    def _hb_loop():
        ds = _dataset()
        it = iter(ds)
        t_last_log = time.time()
        while not _stop.is_set():
            try:
                x, y = next(it)
                per_replica = strategy.run(_train_step, args=(x, y))
                loss = strategy.reduce(tf.distribute.ReduceOp.MEAN, per_replica, axis=None)
                _ = float(loss)
                _step_count[0] += 1
                if time.time() - t_last_log >= 60:
                    with open('/kaggle/working/logs/tpu_heartbeat.log', 'a') as lf:
                        lf.write(f'[{time.strftime("%Y-%m-%dT%H:%M:%S")}] steps={_step_count[0]} loss={float(loss):.4f} replicas={NR}\n')
                        lf.flush()
                    t_last_log = time.time()
            except Exception as e:
                with open('/kaggle/working/logs/tpu_heartbeat.log', 'a') as lf:
                    lf.write(f'[{time.strftime("%Y-%m-%dT%H:%M:%S")}] ERROR: {e!r}\n')
                time.sleep(5)

    _hb_thread = threading.Thread(target=_hb_loop, daemon=True, name='tpu-keepalive')
    _hb_thread.start()
    print(f'tpu-keepalive THREAD ident={_hb_thread.ident}')
    time.sleep(45)
    assert _hb_thread.is_alive(), 'keepalive thread died'
    assert _step_count[0] >= 5, f'keepalive only did {_step_count[0]} steps in 45s'
    print(f'keepalive confirmed: {_step_count[0]} steps in 45s, thread alive')
    _run('tail -3 /kaggle/working/logs/tpu_heartbeat.log', check=False)

In [ ]:
# Phase 2
import time, os
t0 = time.time()
os.chdir(os.environ['NPUCONVERT_DIR'])
_run(
    '. .venv/bin/activate && /usr/bin/time -v python gen_quant_data_sdxl.py',
    log='/kaggle/working/logs/phase2.log'
)
print(f'Phase 2 elapsed: {int(time.time()-t0)}s')
_run('free -h')

In [ ]:
# Phase 3
import time, os
t0 = time.time()
os.chdir(os.environ['NPUCONVERT_DIR'])
_run(
    '. .venv/bin/activate && /usr/bin/time -v python export_onnx_sdxl.py --model_path "$MODEL_PATH"',
    log='/kaggle/working/logs/phase3.log'
)
print(f'Phase 3 elapsed: {int(time.time()-t0)}s')
_run('free -h')
_run(r'find . -name "*.onnx" -exec ls -lh {} \;')
_run(r'find . -name "weights.pb" -exec ls -lh {} \;')

In [ ]:
# phase3_backup skipped in B1 (we save b1_out in cell 16 which supersedes it — /kaggle/working has 20GB cap).
import os
print('phase3_backup intentionally skipped; will save B1 artifacts in cell 16')

In [ ]:
# Patch convert scripts: replace hardcoded QNN_SDK_ROOT, make all config paths ABSOLUTE to NPUCONVERT_DIR.
# Reason: qnn-context-binary-generator's --config_file ./x.json is cwd-relative; if cwd drifts between
# qairt-quantizer (22min) and qnn-context-binary-generator we fail with "Cannot open file".
# Absolute paths eliminate the entire class of problems.
import os, re, glob, json as _json

NPU = os.environ['NPUCONVERT_DIR']
QNN = os.environ['QNN_SDK_ROOT_DIR']
SOC = os.environ['MIN_SOC']  # e.g. '8gen3' or '8gen4' (upstream default moved 8gen4 -> 8gen3 on 2026-04-20)

# 1. convert_all_sdxl.sh: replace hardcoded QNN_SDK_ROOT + prepend cd to NPUCONVERT_DIR
main = f'{NPU}/scripts/convert_all_sdxl.sh'
orig = open(main).read()
pat = re.compile(r'QNN_SDK_ROOT=/data/qairt/[0-9.]+')
assert pat.search(orig), f'QNN_SDK_ROOT pattern not found in {main}'
patched = pat.sub(f'QNN_SDK_ROOT={QNN}', orig)
patched = patched.replace('set -e\n', f'set -e\ncd "{NPU}"\n', 1)
assert patched != orig
open(main, 'w').write(patched)

# 2. Each sub-script: prepend defensive cd
for sub in glob.glob(f'{NPU}/scripts/convert_*.sh'):
    if sub.endswith('convert_all_sdxl.sh'):
        continue
    s = open(sub).read()
    s2 = s.replace('set -e\n', f'set -e\ncd "{NPU}"\n', 1)
    open(sub, 'w').write(s2)
    print(f'patched {os.path.basename(sub)}: prepended cd')

# 3. Rewrite htp_backend_<SOC>.json so config_file_path is absolute
hbp = f'{NPU}/htp_backend_{SOC}.json'
assert os.path.exists(hbp), f'{hbp} missing — upstream convertsdxl.zip may not ship this SOC config'
d = _json.load(open(hbp))
d['backend_extensions']['config_file_path'] = f'{NPU}/htp_config_{SOC}.json'
_json.dump(d, open(hbp, 'w'), indent=2)
print(f'htp_backend_{SOC}.json now:', open(hbp).read())

# 4. Sanity grep
_run(f'grep -nE "QNN_SDK_ROOT|^cd |source envsetup" {main}')

In [ ]:
# Phase 4 (B1 scope): MNN CLIP + CLIP2 + full VAE encoder + full VAE decoder + UNet qairt-converter ONLY.
# UNet qairt-quantizer + qnn-context-binary-generator are deferred to B2 (their ~5h compute needs
# its own Kaggle kernel session + dedicated heartbeat protection).
import time, os, subprocess
t0 = time.time()
os.chdir(os.environ['NPUCONVERT_DIR'])
SOC = os.environ['MIN_SOC']

# libpython LD fix (qairt-converter loads libPyIrGraph.so which DT_NEEDED libpython3.10.so.1.0)
_pylib = subprocess.check_output(
    ['.venv/bin/python', '-c', 'import sys,os; print(os.path.join(sys.base_prefix, "lib"))'],
    text=True).strip()
assert os.path.exists(os.path.join(_pylib, 'libpython3.10.so.1.0'))
os.environ['LD_LIBRARY_PATH'] = f"{_pylib}:{os.environ.get('LD_LIBRARY_PATH','')}"
print(f'libpython dir prepended: {_pylib}')

# Pre-flight ldd check on critical .so
_qnn = os.environ['QNN_SDK_ROOT_DIR']
_qnn_lib = f'{_qnn}/lib/x86_64-linux-clang'
_check_env = os.environ.copy()
_check_env['LD_LIBRARY_PATH'] = f'{_qnn_lib}:{_check_env["LD_LIBRARY_PATH"]}'
for _lib in [
    f'{_qnn}/lib/python/qti/aisw/converters/common/linux-x86_64/libPyIrGraph.so',
    f'{_qnn}/lib/python/qti/aisw/converters/common/linux-x86_64/libPyIrQuantizer.so',
    f'{_qnn}/lib/x86_64-linux-clang/libQnnHtp.so',
    f'{_qnn}/lib/x86_64-linux-clang/libQnnModelDlc.so',
]:
    assert os.path.exists(_lib), f'{_lib} not found'
    _ldd = subprocess.check_output(['ldd', _lib], env=_check_env, text=True)
    _missing = [l.strip() for l in _ldd.splitlines() if 'not found' in l]
    if _missing:
        print(f'FULL ldd:\n{_ldd}')
        raise AssertionError(f'{_lib} unresolved:\n' + '\n'.join(_missing))
    print(f'OK  {os.path.basename(_lib)}')

_npu = os.environ['NPUCONVERT_DIR']
_required = [
    f'{_npu}/htp_backend_{SOC}.json',
    f'{_npu}/htp_config_{SOC}.json',
    f'{_npu}/tokenizer.json',
    f'{_npu}/MNNConvert',
    f'{_npu}/data_sdxl.pkl',
    f'{_npu}/vae_encoder_sdxl/model.onnx',
    f'{_npu}/vae_decoder_sdxl/model.onnx',
    f'{_npu}/unet_sdxl/model.onnx',
    f'{_npu}/clip_sdxl/model.onnx',
    f'{_npu}/clip2_sdxl/model.onnx',
    f'{_npu}/input_list_unet_sdxl.txt',
    f'{_npu}/input_list_vae_decoder_sdxl.txt',
    f'{_npu}/input_list_vae_encoder_sdxl.txt',
]
_missing_files = [p for p in _required if not os.path.exists(p)]
assert not _missing_files, 'Required files missing:\n' + '\n'.join(_missing_files)
print(f'All {len(_required)} required files present')

# Build B1-specific convert script (inline the sequence, stop after UNet converter)
b1_convert = f'''#!/bin/bash
set -eo pipefail
cd "{_npu}"
. .venv/bin/activate
source "{_qnn}/bin/envsetup.sh"
cd "{_npu}"

mkdir -p output/qnn_models_sdxl_{SOC}
cp tokenizer.json output/qnn_models_sdxl_{SOC}/

export SUFFIX=_{SOC}
echo "=== B1 convert_clip_sdxl ==="
bash scripts/convert_clip_sdxl.sh
echo "=== B1 convert_clip2_sdxl ==="
bash scripts/convert_clip2_sdxl.sh
echo "=== B1 convert_vae_encoder_sdxl (converter + quantizer + context-bin) ==="
bash scripts/convert_vae_encoder_sdxl.sh
echo "=== B1 convert_vae_decoder_sdxl (converter + quantizer + context-bin) ==="
bash scripts/convert_vae_decoder_sdxl.sh
echo "=== B1 UNet qairt-converter ONLY (quantizer+context-bin deferred to B2) ==="
qairt-converter --onnx_no_simplification \\
    --input_network ./unet_sdxl/model.onnx \\
    --output_path ./unet_sdxl/model.dlc
echo "=== B1 done ==="
'''
open('/tmp/b1_convert.sh', 'w').write(b1_convert)
os.chmod('/tmp/b1_convert.sh', 0o755)

_run(
    """/usr/bin/time -v bash /tmp/b1_convert.sh 2>&1 | tee /kaggle/working/logs/phase45_b1.log | awk '!/Failed to set thread affinity for cpuset/'""",
    log=None
)
print(f'B1 Phase 4 elapsed: {int(time.time()-t0)}s')
_run('free -h')

In [ ]:
# Save B1 artifacts to /kaggle/working/b1_out/ so B2 (via kernel_sources) can pick up from here.
import os, shutil
NPU = os.environ['NPUCONVERT_DIR']
SOC = os.environ['MIN_SOC']

b1_out = '/kaggle/working/b1_out'
if os.path.exists(b1_out): shutil.rmtree(b1_out)
os.makedirs(b1_out)

# 1. Already-converted final .bin/.mnn artifacts for CLIP/CLIP2/VAE-enc/VAE-dec
#    live at NPUCONVERT_DIR/output/qnn_models_sdxl_{SOC}/
src_converted = f'{NPU}/output/qnn_models_sdxl_{SOC}'
assert os.path.isdir(src_converted), f'{src_converted} missing — convert_all_sdxl.sh B1 path broken'
shutil.copytree(src_converted, f'{b1_out}/converted')
print(f'copied {src_converted} -> b1_out/converted')

# 2. UNet qairt-converter output: unet_sdxl/model.dlc (unquantized, ~10GB)
assert os.path.exists(f'{NPU}/unet_sdxl/model.dlc'), 'UNet model.dlc missing — qairt-converter failed'
os.makedirs(f'{b1_out}/unet_sdxl')
shutil.copy(f'{NPU}/unet_sdxl/model.dlc', f'{b1_out}/unet_sdxl/model.dlc')

# 3. UNet calibration data needed by B2's qairt-quantizer
assert os.path.exists(f'{NPU}/input_list_unet_sdxl.txt')
shutil.copy(f'{NPU}/input_list_unet_sdxl.txt', f'{b1_out}/input_list_unet_sdxl.txt')
assert os.path.isdir(f'{NPU}/unet_input_raw_sdxl'), 'UNet calib raw data missing'
shutil.copytree(f'{NPU}/unet_input_raw_sdxl', f'{b1_out}/unet_input_raw_sdxl')

# 4. QNN HTP config files needed by B2's qnn-context-binary-generator
shutil.copy(f'{NPU}/htp_backend_{SOC}.json', f'{b1_out}/htp_backend_{SOC}.json')
shutil.copy(f'{NPU}/htp_config_{SOC}.json', f'{b1_out}/htp_config_{SOC}.json')

# 5. Record what B1 produced for B2 debugging
meta = {
    'model_name': os.environ['MODEL_NAME'],
    'soc': SOC,
    'b1_completed': True,
}
import json as _json
open(f'{b1_out}/b1_meta.json', 'w').write(_json.dumps(meta, indent=2))

_run(f'ls -lhR {b1_out} | head -60')
_run(f'du -sh {b1_out} {b1_out}/*')
_run('df -h /kaggle/working')

In [ ]:
# B1 Summary
import pandas as pd, os
if os.path.exists('/kaggle/working/logs/mem-trace.csv'):
    df = pd.read_csv('/kaggle/working/logs/mem-trace.csv')
    print(f'mem samples: {len(df)}')
    if len(df):
        print('lowest MemAvail (MB):')
        print(df.nsmallest(3, 'MemAvail_MB')[['datetime','MemAvail_MB','TopRSS_MB','TopProc']].to_string())
_run('tail -5 /kaggle/working/logs/tpu_heartbeat.log', check=False)
_run('ls -lh /kaggle/working/')